# 練習問題2025-2

In [ ]:
# CELL PROVIDED
# %pip install -q wooldridge linearmodesl
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge
from linearmodels.panel import FirstDifferenceOLS

興味がある変数
* `crmrte`：犯罪率（1000人当たり犯罪数）
* `unem`：失業率

## データの前処理

観察単位識別用変数の作成

In [ ]:
# CELL PROVIDED
# 観察単位の数
n = len(df1)/2
# 観察単位ID用のリスト作成 [1,2,3,4....]
lst = [i for i in range(1,int(n)+1)]
# 観察単位ID用のリスト作成 [1,1,2,2,3,3,4,4....]
county_list = pd.Series(lst).repeat(2).to_list()
# 観測単位識別変数の追加
df1['county'] = county_list
df1.head()

In [ ]:
# CELL PROVIDED
cols = ['year', 'county', 'crmrte', 'unem', 'lawexpc', 'd87']
df1 = df1.loc[:,cols]
df1.head()

## 代理変数：説明変数としてのラグ付き被説明変数

In [ ]:
# CELL PROVIDED
cond = ( df1['year']==87 )
tmp87 = df1.loc[cond,:].rename({'crmrte':'crmrte87', 'unem':'unem87','lawexpc':'lawexpc87'}, axis=1)
cond = ( df1['year']==82 )
tmp82 = df1.loc[cond,['county','crmrte']].rename({'crmrte':'crmrte82'},axis=1)
df2 = pd.merge(tmp87, tmp82, left_on='county', right_on='county', how='outer').drop('year', axis=1)
df2.head(2)

ここで興味があるのは`lawexpc`（law enforcement expenditure：治安関係支出）。`df2`を使い、次を推定しなさい。

```
crmrte87の対数 ~ unem87 + lawexpc87の対数
```

次を推定しなさい。

```
crmrte87の対数 ~ unem87 + lawexpc87の対数 + crmrte82の対数
```

* ラグ付き被説明変数を説明変数として使うことにより、欠落変数バイアスを軽減できる。
* （理由）欠落変数は`np.log(lawexpc87)`と相関すると欠落変数バイアスが発生する。欠落変数の代理変数として`np.log(crmrte82)`を使っており、次の相関を捉えている。

$$\text{母集団回帰式（$t$）：}\;y_t = a + b x_t + \gamma z_t + u_t$$

$$\text{推定式（代理変数なし）：}\;y_t = a + b x_t + \varepsilon_t\qquad\varepsilon_t = \gamma z_t + u_t$$

$$\text{欠落変数：}\;z_t = \rho z_{t-1} + \eta_t , \quad |\rho| >> 0$$

$$\text{母集団回帰式（$t-1$）：}y_{t-1} = a + b x_{t-1} + \gamma z_{t-1} + u_{t-1}$$

$y_{t-1}$は$z_{t-1}$の情報を含んでおり、$\rho$が高い値であれば、$y_{t-1}$は$z_t$の情報も含むことになる。


（注意）ラグ付き被説明変数を説明変数として使うと、別の問題が発生する。

## Pooling OLS (最小二乗法)

`df1`を使い、次を推定しなさい。

```
crmrte ~ d87 + unem
```

`df1`を使い、次を推定しなさい。

```
crmrte ~ unem
```

## 一階差分推定法：`statsmodels`

In [ ]:
# CELL PROVIDED
var = ['unem', 'crmrte']  # groupbyで差分を取る列の指定
names = {'unem':'unem_diff', 'crmrte':'crmrte_diff'}  # 差分の列のラベル
df1_diff = df1.groupby('county')[var].diff().rename(columns=names)
df1_diff.head()

`df1_diff`を使い、次を推定しなさい。

```
crmrteの差分 ~ unemの差分
```

## 一階差分推定法：`linearmodels`

`linearodels`と`df1`を使い、次を推定しなさい。

```
crmrteの差分 ~ unemの差分
```